In [ ]:
!pip install sentence-transformers faiss-cpu requests

In [ ]:
text = """
Transformers use self-attention mechanisms.
RAG stands for Retrieval Augmented Generation.
Vector databases store embeddings and perform similarity search.
PCA reduces dimensionality by projecting onto principal components.
"""

with open("notes.txt", "w") as f:
    f.write(text)

print("File created.")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load embedding model (runs locally inside Colab)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Read file
with open("notes.txt", "r") as f:
    text = f.read()

# Split into chunks (simple version)
chunks = text.strip().split("\n")

# Convert text to embeddings
embeddings = model.encode(chunks)

# Convert to numpy float32
embeddings = np.array(embeddings).astype("float32")

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Embeddings created and stored in vector database.")


In [ ]:
def retrieve(query, top_k=2):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

# Test it
print(retrieve("What is RAG?"))


In [ ]:
import requests

HF_TOKEN = "hf_***********************"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "How many G's are in huggingface?"
        }
    ],
    "max_tokens": 50
}


response = requests.post(
    "https://router.huggingface.co/v1/chat/completions",
    headers=headers,
    json=payload
)

print("Status:", response.status_code)
print(response.text)


In [ ]:
import requests

HF_TOKEN = "hf_***********************"  # paste again locally (not publicly)

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}




def generate_answer(query):
    payload = {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "system", "content": "You are an AI assistant specialized in machine learning and AI."}, # to make it smarter and to direct its function i guess
            {"role": "user", "content": query}
        ],
        "max_tokens": 150
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        return f"Error: {response.text}"

    return response.json()["choices"][0]["message"]["content"]


print(generate_answer("Explain RAG in simple terms"))


In [ ]:
print(generate_answer(
    "In the field of machine learning, deep learning, and artificial intelligence, what is Retrieval Augmented Generation (RAG)? Explain"
))
